# 03 — BERT Fine-tuning

**Subject:** NLP — Topic 2 (Isaac González)  
**Environment:** Google Colab — T4 GPU required  
**Goal:** Fine-tune BERT on travel Twitter captions.

## Why BERT over classical methods?
Classical TF-IDF treats each word independently. BERT generates
**contextual embeddings** — the same word gets a different vector
depending on its context:
- `"sick view from the hotel"` → positive (slang)
- `"feeling sick at the airport"` → negative (illness)

TF-IDF cannot distinguish these cases. BERT can.

## Architecture
- Base model: `bert-base-uncased` (110M parameters)
- Added: linear classification head on the `[CLS]` token
- Trained with: AdamW optimiser, lr=2e-5, 3 epochs

## Expected results
- F1-macro: ~0.72 (+11 points over Logistic Regression)
- Training time: ~20 minutes on T4 GPU

## Cell 1 — Install dependencies (Colab only)

In [ ]:
!pip install transformers datasets accelerate -q

## Cell 2 — Mount Google Drive

The dataset CSV is stored in Google Drive.
Accept the permissions when prompted.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 3 — Load dataset

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/multimodal_emotion/data/labels.csv')
print(f"Dataset shape: {df.shape}")
print(df['label'].value_counts())
df.head()

## Cell 4 — Imports and GPU check

⚠️ Make sure GPU is enabled: Runtime → Change runtime type → T4 GPU  
Expected output: `Device: cuda`

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import (BertTokenizer, BertForSequenceClassification,
                          get_linear_schedule_with_warmup)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.preprocessing import LabelEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
# If this shows 'cpu', enable GPU in Runtime settings before continuing.

## Cell 5 — CaptionDataset class

This PyTorch Dataset tokenises each caption using BERT's tokeniser.
The tokeniser:
- Splits text into subword tokens
- Adds special tokens: `[CLS]` at the start, `[SEP]` at the end
- Pads/truncates to `max_length=128` tokens
- Returns `input_ids` and `attention_mask`

In [ ]:
class CaptionDataset(Dataset):
    """PyTorch Dataset that tokenises tweet captions for BERT."""

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

## Cell 6 — Data preparation

Same `random_state=42` and `test_size=0.2` as all other modules
to ensure a fair comparison.

In [ ]:
le     = LabelEncoder()
labels = le.fit_transform(df['label'].tolist())
texts  = df['text'].tolist()
print(f"Classes: {le.classes_}")

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
train_ds  = CaptionDataset(X_train, y_train, tokenizer)
test_ds   = CaptionDataset(X_test,  y_test,  tokenizer)
train_dl  = DataLoader(train_ds, batch_size=16, shuffle=True)
test_dl   = DataLoader(test_ds,  batch_size=32)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")

## Cell 7 — Model and optimiser

We use `BertForSequenceClassification` which adds a linear layer
on top of the `[CLS]` token representation.

**Optimiser:** AdamW with weight_decay=0.01 (L2 regularisation).  
**Scheduler:** Linear warmup (5% of steps) then linear decay.

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(le.classes_)
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_dl) // 5,   # 5% warmup
    num_training_steps=len(train_dl) * 3   # 3 epochs total
)
print("Model ready.")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## Cell 8 — Training (3 epochs)

⏱️ **Expected time: ~20 minutes on T4 GPU**

Expected loss progression: 0.84 → 0.55 → 0.38  
Gradient clipping (max_norm=1.0) prevents exploding gradients.

In [ ]:
train_losses = []

for epoch in range(3):
    model.train()
    total_loss = 0

    for i, batch in enumerate(train_dl):
        optimizer.zero_grad()
        out = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['label'].to(device)
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # gradient clipping
        optimizer.step()
        scheduler.step()
        total_loss += out.loss.item()

        if i % 30 == 0:
            print(f"Epoch {epoch+1} | Batch {i}/{len(train_dl)} | "
                  f"Loss: {out.loss.item():.4f}")

    avg = total_loss / len(train_dl)
    train_losses.append(avg)
    print(f"\n>>> Epoch {epoch+1} complete — Average loss: {avg:.4f}\n")

## Cell 9 — Evaluation on test set

In [ ]:
model.eval()
all_preds, all_probas = [], []

with torch.no_grad():
    for batch in test_dl:
        out   = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        probs = torch.softmax(out.logits, dim=1).cpu().numpy()
        all_probas.extend(probs.tolist())
        all_preds.extend(np.argmax(probs, axis=1).tolist())

print("=== BERT Fine-tuned ===")
print(classification_report(y_test, all_preds, target_names=le.classes_))
print(f"F1 macro: {f1_score(y_test, all_preds, average='macro'):.4f}")

## Cell 10 — Save results to Drive

After running this cell, download `metrics_bert.json` from Drive
and place it in `results/` in your local repository.

In [ ]:
import json, os

os.makedirs('/content/drive/MyDrive/multimodal_emotion/results', exist_ok=True)

results = {
    'f1_macro':      f1_score(y_test, all_preds, average='macro'),
    'probas':        all_probas,   # saved for late fusion
    'train_losses':  train_losses,
    'label_encoder': le.classes_.tolist()
}

with open('/content/drive/MyDrive/multimodal_emotion/results/metrics_bert.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to Drive.")
print("Download metrics_bert.json and place it in results/ in your local repo.")

## Cell 11 — Training loss curve

A decreasing loss curve confirms the model is learning.
The curve is saved to Drive and should be added to `results/figures/`.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(range(1, 4), train_losses, marker='o', color='#7c3aed', linewidth=2)
plt.title('Training Loss — BERT Fine-tuning')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks([1, 2, 3])
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/multimodal_emotion/results/bert_loss_curve.png',
            dpi=150)
plt.show()
print("Loss curve saved to Drive.")